<a href="https://colab.research.google.com/github/oluwapelumi1/ACCESS-SAP-DATA-Science-Track/blob/main/Week2_Session4_Data_Cleaning_And_Feature_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Week2_session4_Data_Cleaning_And_Feature_Creation**

**objective**:

---
using the dataset [Montgomery County liquor sales](https://www.kaggle.com/datasets/samanfatima7/warehouse-and-retail-sales-montgomery-county)


*   Handle missing values and duplicates.

*   Create at least two new derived columns (e.g., performance ratios or total campaign success).


*  Save the cleaned dataset as a new CSV and commit to GitHub.












In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
#Read the dataset
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Warehouse_and_Retail_Sales.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#Get an idea of the dataset
df.head()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
0,2020,1,REPUBLIC NATIONAL DISTRIBUTING CO,100009,BOOTLEG RED - 750ML,WINE,0.00,0.0,2.0
1,2020,1,PWSWN INC,100024,MOMENT DE PLAISIR - 750ML,WINE,0.00,1.0,4.0
2,2020,1,RELIABLE CHURCHILL LLLP,1001,S SMITH ORGANIC PEAR CIDER - 18.7OZ,BEER,0.00,0.0,1.0
3,2020,1,LANTERNA DISTRIBUTORS INC,100145,SCHLINK HAUS KABINETT - 750ML,WINE,0.00,0.0,1.0
4,2020,1,DIONYSOS IMPORTS INC,100293,SANTORINI GAVALA WHITE - 750ML,WINE,0.82,0.0,0.0


**MISSING VALUES**

In [4]:
#ADDITION OF MISSING VALUES
df.isna().sum()

,0
YEAR,0
MONTH,0
SUPPLIER,167
ITEM CODE,0
ITEM DESCRIPTION,0
ITEM TYPE,1
RETAIL SALES,3
RETAIL TRANSFERS,0
WAREHOUSE SALES,0


In [5]:
print("Missing values per column:")
print(df.isnull().sum())


Missing values per column:
YEAR                  0
MONTH                 0
SUPPLIER            167
ITEM CODE             0
ITEM DESCRIPTION      0
ITEM TYPE             1
RETAIL SALES          3
RETAIL TRANSFERS      0
WAREHOUSE SALES       0
dtype: int64


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307645 entries, 0 to 307644
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YEAR              307645 non-null  int64  
 1   MONTH             307645 non-null  int64  
 2   SUPPLIER          307478 non-null  object 
 3   ITEM CODE         307645 non-null  object 
 4   ITEM DESCRIPTION  307645 non-null  object 
 5   ITEM TYPE         307644 non-null  object 
 6   RETAIL SALES      307642 non-null  float64
 7   RETAIL TRANSFERS  307645 non-null  float64
 8   WAREHOUSE SALES   307645 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 21.1+ MB


In [12]:
df.tail()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
307640,2020,9,LEGENDS LTD,99753,DUTCHESS DE BOURGOGNE NR - 750ML,BEER,0.00,0.0,5.00
307641,2020,9,ANHEUSER BUSCH INC,9997,HOEGAARDEN 4/6NR - 12OZ,BEER,66.12,37.0,240.75
307642,2020,9,COASTAL BREWING COMPANY LLC,99970,DOMINION OAK BARREL STOUT 4/6 NR - 12OZ,BEER,2.25,0.0,0.00
307643,2020,9,BOSTON BEER CORPORATION,99990,SAM ADAMS SUMMER VARIETY 12PK NR,BEER,20.50,0.0,0.00
307644,2020,9,NaN,WC,WINE CREDIT,REF,0.00,0.0,-70.00


In [14]:
df_clean=df.copy()

In [27]:
# SUPPLIER - 167 MISSING
df_clean['SUPPLIER'] = df_clean['SUPPLIER'].fillna('Not Specified')
# ITEM TYPE - 1 MISSING
df_clean['ITEM TYPE'] = df_clean['ITEM TYPE'].fillna('Not Specified')
# RETAIL SALES - 3 MISSING
df_clean = df_clean.dropna(subset=['RETAIL SALES'])


# Step 3: Verify
print(f"\nOriginal shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"\nRemaining missing values:")
print(df_clean.isnull().sum())


Original shape: (307645, 9)
Cleaned shape: (307642, 9)

Remaining missing values:
YEAR                0
MONTH               0
SUPPLIER            0
ITEM CODE           0
ITEM DESCRIPTION    0
ITEM TYPE           0
RETAIL SALES        0
RETAIL TRANSFERS    0
WAREHOUSE SALES     0
dtype: int64


## **Check for Duplicates**

In [28]:
df_clean.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
307640,False
307641,False
307642,False
307643,False


In [38]:
# Check for duplicate rows
duplicates_count = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")

if duplicates_count > 0:
    # Drop duplicate rows
    df_clean_no_duplicates = df_clean.drop_duplicates()
    print(f"Shape after dropping duplicates: {df_clean_no_duplicates.shape}")
else:
    print("No duplicates found.")
    df_clean_no_duplicates = df_clean.copy()

Number of duplicate rows: 0
No duplicates found.



###**Feature Creation**

In [39]:
# Categorize RETAIL SALES into spending levels
def categorize_amount(retail_sales):
    if retail_sales < -6:
        return 'Small'
    elif retail_sales < 30:
        return 'Medium'
    else:
        return 'Large'

df_clean['SALES_level'] = df_clean['RETAIL SALES'].apply(categorize_amount)

print(df_clean['SALES_level'].value_counts())

SALES_level
Medium    291006
Large      16635
Small          1
Name: count, dtype: int64


**Creating Boolean Flags**

In [41]:
df_clean['is_high_value'] = df_clean['RETAIL SALES'] > 30

# Flag monthly transactions (original line, kept for completeness)
df_clean['MONTHLY'] = df_clean['MONTH'].isin([1, 9])

In [42]:
df_clean['TOTAL_SALES'] = df_clean['RETAIL SALES'] + df_clean['WAREHOUSE SALES']
print(df_clean[['RETAIL SALES', 'WAREHOUSE SALES', 'TOTAL_SALES']].head())

   RETAIL SALES  WAREHOUSE SALES  TOTAL_SALES
0          0.00              2.0         2.00
1          0.00              4.0         4.00
2          0.00              1.0         1.00
3          0.00              1.0         1.00
4          0.82              0.0         0.82


##**Saving Your Cleaned Data**

In [43]:
df_clean.to_csv('cleaned_warehouse_retail_sales.csv', index=False)
print("Cleaned dataset saved to 'cleaned_warehouse_retail_sales.csv'")

Cleaned dataset saved to 'cleaned_warehouse_retail_sales.csv'
